# 23 — Geometric Curvature of the K_nm Scale Manifold

K_nm defines a natural distance on the space of observation scales:
d(i,j) = -log K[i,j]. This distance matrix defines a weighted graph
whose discrete curvature can be computed.

Tests:
1. Does scalar curvature peak at K_c (synchronisation transition)?
2. Is the manifold hyperbolic (as predicted by log-frequency spacing)?
3. Does torsion from asymmetric TE data break even/odd parity?

Methods: Ollivier-Ricci curvature, Forman-Ricci curvature, graph Laplacian spectral analysis.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np

sys.path.insert(
    0,
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/src",
)
from scpn_quantum_control.bridge.knm_hamiltonian import OMEGA_N_16, build_knm_paper27
from scpn_quantum_control.hardware.classical import classical_kuramoto_reference

print("Imports OK")

## 1. K_nm Distance Matrix and Graph Laplacian

In [ ]:
def knm_to_distance(K):
    """Convert coupling matrix to distance: d(i,j) = -log(K[i,j]).
    Diagonal set to 0. Clip K > 0 to avoid log(0)."""
    K_safe = np.clip(K, 1e-10, None)
    D = -np.log(K_safe)
    np.fill_diagonal(D, 0)
    return D


def graph_laplacian(W, normalised=False):
    """Graph Laplacian L = D - W. Optionally normalised: L_norm = I - D^{-1/2} W D^{-1/2}."""
    D = np.diag(np.sum(W, axis=1))
    L = D - W
    if normalised:
        D_inv_sqrt = np.diag(1.0 / np.sqrt(np.diag(D) + 1e-10))
        L = D_inv_sqrt @ L @ D_inv_sqrt
    return L


def directed_laplacian(W):
    """Directed graph Laplacian for asymmetric W.
    L_dir = D_out - W where D_out = diag(sum of outgoing weights)."""
    D_out = np.diag(np.sum(W, axis=1))
    return D_out - W


# Build K_nm for various sizes
K16 = build_knm_paper27(L=16)
D16 = knm_to_distance(K16)
L16 = graph_laplacian(K16)

# Spectrum of Laplacian
eigvals_L = np.sort(np.linalg.eigvalsh(L16))
fiedler = eigvals_L[1]  # algebraic connectivity

print(f"K_nm shape: {K16.shape}")
print(f"Distance range: [{D16[D16 > 0].min():.3f}, {D16.max():.3f}]")
print(f"Fiedler value (algebraic connectivity): {fiedler:.4f}")
print(f"Laplacian spectrum: {eigvals_L[:5].round(4)}")

## 2. Forman-Ricci Curvature

Forman curvature on an edge (i,j) of a weighted graph:
F(i,j) = w(i,j) * [w(i)/w(i,j) + w(j)/w(i,j) - sum_k (w(i,k)/sqrt(w(i,j)*w(i,k))) ...]

Simplified form for weighted graphs without higher-dimensional cells:
F(i,j) = 4 - d_i - d_j + sum of parallel transport corrections

In [ ]:
def forman_ricci_curvature(W, threshold=0.01):
    """Compute Forman-Ricci curvature for each edge of weighted graph W.

    For edge (i,j) with weight w_ij:
    F(i,j) = w_ij * (2/w_ij - sum_{k~i, k!=j} w_ij/(sqrt(w_ij * w_ik))
                              - sum_{k~j, k!=i} w_ij/(sqrt(w_ij * w_jk)))

    Returns dict of {(i,j): curvature}.
    """
    n = W.shape[0]
    curvatures = {}
    for i in range(n):
        for j in range(i + 1, n):
            w_ij = W[i, j]
            if w_ij < threshold:
                continue

            # Neighbours of i (excluding j)
            sum_i = 0
            for k in range(n):
                if k in (i, j):
                    continue
                w_ik = W[i, k]
                if w_ik > threshold:
                    sum_i += w_ij / np.sqrt(w_ij * w_ik)

            # Neighbours of j (excluding i)
            sum_j = 0
            for k in range(n):
                if k in (i, j):
                    continue
                w_jk = W[j, k]
                if w_jk > threshold:
                    sum_j += w_ij / np.sqrt(w_ij * w_jk)

            F = w_ij * (2.0 / w_ij - sum_i - sum_j)
            curvatures[(i, j)] = F

    return curvatures


def mean_forman_curvature(W, threshold=0.01):
    """Mean Forman-Ricci curvature (scalar curvature analogue)."""
    curvs = forman_ricci_curvature(W, threshold)
    if not curvs:
        return 0.0
    return float(np.mean(list(curvs.values())))


# Compute for K_nm at different sizes
for L in [4, 6, 8, 12, 16]:
    K = build_knm_paper27(L=L)
    mfc = mean_forman_curvature(K)
    print(f"L={L:2d}: mean Forman curvature = {mfc:.4f}")

## 3. Ollivier-Ricci Curvature

Ollivier-Ricci curvature uses optimal transport (Wasserstein distance)
between probability measures on neighbouring nodes:
kappa(i,j) = 1 - W_1(mu_i, mu_j) / d(i,j)

Positive = sphere-like (convergent geodesics)
Zero = flat
Negative = hyperbolic (divergent geodesics)

In [ ]:
def ollivier_ricci_curvature(W, alpha=0.5, threshold=0.01):
    """Compute Ollivier-Ricci curvature for each edge.

    mu_i = alpha * delta_i + (1-alpha) * uniform over neighbours of i
    kappa(i,j) = 1 - W_1(mu_i, mu_j) / d(i,j)

    Uses scipy linear_sum_assignment for optimal transport.
    """
    n = W.shape[0]
    # Distance matrix from weights
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i == j:
                D[i, j] = 0
            elif W[i, j] > threshold:
                D[i, j] = 1.0 / W[i, j]  # inverse weight as distance
            else:
                D[i, j] = 100.0  # large distance for disconnected

    # Shortest paths via Floyd-Warshall
    from scipy.sparse.csgraph import shortest_path

    D_sp = shortest_path(D, directed=False)

    curvatures = {}
    for i in range(n):
        for j in range(i + 1, n):
            if W[i, j] < threshold:
                continue

            d_ij = D_sp[i, j]
            if d_ij <= 0 or np.isinf(d_ij):
                continue

            # Build measures mu_i and mu_j
            # mu_i: alpha on i, (1-alpha) uniform on neighbours
            nbrs_i = [k for k in range(n) if W[i, k] > threshold and k != i]
            nbrs_j = [k for k in range(n) if W[j, k] > threshold and k != j]

            if not nbrs_i or not nbrs_j:
                continue

            # Support of mu_i: {i} + nbrs_i
            supp_i = [i] + nbrs_i
            mass_i = np.zeros(len(supp_i))
            mass_i[0] = alpha
            mass_i[1:] = (1 - alpha) / len(nbrs_i)

            supp_j = [j] + nbrs_j
            mass_j = np.zeros(len(supp_j))
            mass_j[0] = alpha
            mass_j[1:] = (1 - alpha) / len(nbrs_j)

            # Cost matrix
            cost = np.zeros((len(supp_i), len(supp_j)))
            for a_idx, a_node in enumerate(supp_i):
                for b_idx, b_node in enumerate(supp_j):
                    cost[a_idx, b_idx] = D_sp[a_node, b_node]

            # Approximate W_1 via greedy transport (exact for small n)
            # Use earth mover's distance approximation
            # Flatten to 1D Wasserstein on graph distances
            # More accurate: solve transport problem directly
            # For small graphs, direct LP
            w1 = _wasserstein_graph(mass_i, mass_j, cost)

            kappa = 1.0 - w1 / d_ij
            curvatures[(i, j)] = float(kappa)

    return curvatures


def _wasserstein_graph(p, q, C):
    """Compute Wasserstein-1 distance via linear programming."""
    from scipy.optimize import linprog

    m, n = len(p), len(q)
    # Variables: T[i,j] (transport plan), flattened
    c = C.flatten()

    # Constraints: sum_j T[i,j] = p[i], sum_i T[i,j] = q[j]
    A_eq = np.zeros((m + n, m * n))
    for i in range(m):
        A_eq[i, i * n : (i + 1) * n] = 1
    for j in range(n):
        for i in range(m):
            A_eq[m + j, i * n + j] = 1
    b_eq = np.concatenate([p, q])

    bounds = [(0, None)] * (m * n)
    result = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
    if result.success:
        return result.fun
    return float("inf")


# Compute Ollivier-Ricci for K_nm
orc = ollivier_ricci_curvature(K16, alpha=0.5)
orc_vals = list(orc.values())
print("Ollivier-Ricci curvature on K_nm (16x16):")
print(f"  Mean: {np.mean(orc_vals):.4f}")
print(f"  Min:  {np.min(orc_vals):.4f}")
print(f"  Max:  {np.max(orc_vals):.4f}")
print(f"  Std:  {np.std(orc_vals):.4f}")
n_neg = sum(1 for v in orc_vals if v < 0)
n_pos = sum(1 for v in orc_vals if v > 0)
print(f"  Negative (hyperbolic): {n_neg}/{len(orc_vals)}")
print(f"  Positive (spherical):  {n_pos}/{len(orc_vals)}")

## 4. Curvature vs Coupling Strength — Does Curvature Peak at K_c?

In [ ]:
# Sweep K_scale and compute curvature + order parameter R
K_scales = np.linspace(0.1, 5.0, 25)
N = 8  # tractable size
K_base = build_knm_paper27(L=N)
omega = OMEGA_N_16[:N]

forman_vs_K = []
fiedler_vs_K = []
R_vs_K = []

for K_scale in K_scales:
    K_scaled = K_base * K_scale

    # Forman curvature
    mfc = mean_forman_curvature(K_scaled, threshold=0.001)
    forman_vs_K.append(mfc)

    # Fiedler value
    L = graph_laplacian(K_scaled)
    eigvals = np.sort(np.linalg.eigvalsh(L))
    fiedler_vs_K.append(float(eigvals[1]))

    # Order parameter from Kuramoto
    ref = classical_kuramoto_reference(N, t_max=20.0, dt=0.05, K=K_scaled, omega=omega)
    R_final = float(ref["R"][-1])
    R_vs_K.append(R_final)

# Find K_c (steepest R increase)
dR = np.diff(R_vs_K)
K_c_idx = np.argmax(dR)
K_c_est = float(K_scales[K_c_idx])

# Find curvature extremum
forman_arr = np.array(forman_vs_K)
curv_peak_idx = np.argmin(forman_arr)  # most negative = most hyperbolic
K_curv_peak = float(K_scales[curv_peak_idx])

print(f"Estimated K_c (max dR/dK): {K_c_est:.2f}")
print(f"Forman curvature extremum at K_scale: {K_curv_peak:.2f}")
print(
    f"Match: {'YES' if abs(K_c_est - K_curv_peak) < 0.5 else 'NO'} (delta={abs(K_c_est - K_curv_peak):.2f})"
)
print()

# Print table
print(f"{'K_scale':>8} {'R':>8} {'Forman':>10} {'Fiedler':>10}")
for i in range(len(K_scales)):
    marker = " <-- K_c" if i == K_c_idx else ""
    print(
        f"{K_scales[i]:8.2f} {R_vs_K[i]:8.4f} {forman_vs_K[i]:10.4f} {fiedler_vs_K[i]:10.4f}{marker}"
    )

## 5. Torsion from Asymmetric Transfer Entropy

Directed coupling creates torsion in the connection.
T^k_{ij} = K[i,j] - K[j,i] (antisymmetric part).
Torsion magnitude indicates directionality of information flow.

In [ ]:
# Build directed K_nm from TE asymmetry data (notebook 19 results)
# TE asymmetry values from session state:
# BP→alpha 0.65-0.81, resp→alpha 0.58-0.78, delta→beta 0.39-0.59, ECG→alpha 0.35-0.63
# Grand mean asymmetry: 0.361

# For a 16x16 matrix, simulate directional bias:
# Autonomic (L1-L3) → cortical (L11-L16) is stronger than reverse
K_sym = build_knm_paper27(L=16)

# Create directed version with TE-like asymmetry
# Lower indices (autonomic scales) drive upper (cortical scales)
K_dir = K_sym.copy()
mean_asym = 0.361
for i in range(16):
    for j in range(i + 1, 16):
        # Lower index → higher index gets boost
        boost = mean_asym * K_sym[i, j]
        K_dir[i, j] = K_sym[i, j] + boost  # i→j (upward)
        K_dir[j, i] = K_sym[j, i] - boost  # j→i (downward, weakened)
        K_dir[j, i] = max(K_dir[j, i], 0.001)  # keep positive

# Torsion tensor: antisymmetric part
torsion = K_dir - K_dir.T
torsion_magnitude = np.linalg.norm(torsion, "fro")
print(f"Torsion Frobenius norm: {torsion_magnitude:.4f}")
print(f"Symmetric part norm: {np.linalg.norm(K_sym, 'fro'):.4f}")
print(f"Torsion / symmetric ratio: {torsion_magnitude / np.linalg.norm(K_sym, 'fro'):.4f}")

# Directed Laplacian spectrum
L_dir = directed_laplacian(K_dir)
eigvals_dir = np.sort(np.abs(np.linalg.eigvals(L_dir)))
fiedler_dir = eigvals_dir[1]
print(f"\nDirected Fiedler value: {fiedler_dir:.4f}")
print(f"Symmetric Fiedler value: {fiedler:.4f}")
print(f"Direction speeds up connectivity: {'YES' if fiedler_dir > fiedler else 'NO'}")

# Even/odd parity of torsion
# The torsion matrix has structure: upper triangle positive (upward flow dominates)
# This breaks the even/odd symmetry
n_pos_torsion = np.sum(torsion > 0)
n_neg_torsion = np.sum(torsion < 0)
print(f"\nTorsion positive entries: {n_pos_torsion}")
print(f"Torsion negative entries: {n_neg_torsion}")
print(f"Parity symmetry: {'BROKEN' if n_pos_torsion != n_neg_torsion else 'PRESERVED'}")

## 6. Hyperbolicity Test

A graph is delta-hyperbolic if for all quadruples (a,b,c,d):
the two largest sums of d(a,b)+d(c,d), d(a,c)+d(b,d), d(a,d)+d(b,c)
differ by at most 2*delta.

Small delta = tree-like (hyperbolic). Large delta = not hyperbolic.

In [ ]:
def gromov_hyperbolicity(D):
    """Compute Gromov delta-hyperbolicity of distance matrix D.

    For all quadruples, compute the four-point condition.
    Returns delta (smallest value such that the graph is delta-hyperbolic).
    """
    n = D.shape[0]
    max_delta = 0.0
    for a in range(n):
        for b in range(a + 1, n):
            for c in range(b + 1, n):
                for d in range(c + 1, n):
                    s1 = D[a, b] + D[c, d]
                    s2 = D[a, c] + D[b, d]
                    s3 = D[a, d] + D[b, c]
                    sums = sorted([s1, s2, s3])
                    delta = (sums[2] - sums[1]) / 2.0
                    max_delta = max(max_delta, delta)
    return max_delta


# Compute hyperbolicity for K_nm distance matrix
from scipy.sparse.csgraph import shortest_path

# Use inverse weight as distance for shortest paths
D_weight = np.where(K16 > 0.01, 1.0 / K16, 100.0)
np.fill_diagonal(D_weight, 0)
D_sp = shortest_path(D_weight, directed=False)

delta = gromov_hyperbolicity(D_sp)
diameter = np.max(D_sp[np.isfinite(D_sp)])
relative_delta = delta / diameter

print(f"Gromov delta-hyperbolicity: {delta:.4f}")
print(f"Graph diameter: {diameter:.4f}")
print(f"Relative delta (delta/diameter): {relative_delta:.4f}")
print()
if relative_delta < 0.1:
    print("Graph is strongly hyperbolic (tree-like)")
elif relative_delta < 0.25:
    print("Graph has moderate hyperbolicity")
else:
    print("Graph is NOT hyperbolic (not tree-like)")

# Compare with pure tree and complete graph
print("\nFor reference: tree delta=0, complete graph delta~diameter/2")

In [ ]:
# Save all results
results = {
    "experiment": "Geometric curvature of K_nm scale manifold",
    "forman_curvature_16": mean_forman_curvature(K16),
    "ollivier_ricci_16": {
        "mean": round(float(np.mean(orc_vals)), 4),
        "min": round(float(np.min(orc_vals)), 4),
        "max": round(float(np.max(orc_vals)), 4),
        "n_negative": n_neg,
        "n_positive": n_pos,
    },
    "curvature_vs_coupling": {
        "K_scales": [round(float(k), 2) for k in K_scales],
        "R": [round(float(r), 4) for r in R_vs_K],
        "forman": [round(float(f), 4) for f in forman_vs_K],
        "fiedler": [round(float(f), 4) for f in fiedler_vs_K],
        "K_c_estimate": K_c_est,
        "curvature_extremum_K": K_curv_peak,
    },
    "torsion": {
        "frobenius_norm": round(torsion_magnitude, 4),
        "symmetric_norm": round(float(np.linalg.norm(K_sym, "fro")), 4),
        "ratio": round(torsion_magnitude / np.linalg.norm(K_sym, "fro"), 4),
        "directed_fiedler": round(float(fiedler_dir), 4),
        "symmetric_fiedler": round(float(fiedler), 4),
    },
    "hyperbolicity": {
        "gromov_delta": round(float(delta), 4),
        "diameter": round(float(diameter), 4),
        "relative_delta": round(float(relative_delta), 4),
    },
    "fiedler_value": round(float(fiedler), 4),
}

out_path = Path(
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/results/geometric_curvature_2026-03-29.json"
)
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"Results saved to {out_path}")